# AST-based Pattern Matching

In GitHub Codespaces, select the Python kernel at `/opt/jupyter/bin/python`.
Run the cells in order. If the imports fail, run
`%pip install tree-sitter tree-sitter-c` in a new code cell.

Implement the patterns yourself by inspecting nodes and traversing their
children. The exercise functions contain `pass`: replace it with your code.
Until then, calls return `None` or print nothing.

We use Tree-sitter's syntax tree as our AST representation. It also includes
punctuation, such as braces and semicolons.

In [ ]:
#pip install tree_sitter
import tree_sitter_c as tsc
from tree_sitter import Language, Parser

In [ ]:
C_LANGUAGE = Language(tsc.language())
parser = Parser(C_LANGUAGE)


program = """
void sendMessage(int, const char *);

int foo() {
   return 0;
}

int main() {
    sendMessage(foo(), "My message");
}
"""

tree = parser.parse(bytes(program, "utf-8"))

## Recursive Traversal

`print_tree` prints the current node, then visits each child recursively.
The indentation shows its depth; the leaves also show their source text.

In [ ]:
def print_tree(node, depth=0):
    prefix = " |" * depth
    if not node.children:
        suffix = '|' + str(node.text, encoding="utf-8") + '|'
    else:
        suffix = ""
    print(f"{prefix} <{node.grammar_name}> {suffix}")
    for child in node.children:
        print_tree(child, depth=depth+1)

print_tree(tree.root_node)

Useful node operations:

- `node.grammar_name`: the kind of node.
- `node.children`: children in source order, including punctuation.
- `node.child_by_field_name("consequence")`: a child with that field name,
  or `None` when absent.
- `node.text`: source text covered by the node, as byte-string.
- `node.start_point.row`: the node's line number, starting at 0.

Use `print_tree` to inspect the examples when needed.
See the [Node reference](https://tree-sitter.github.io/py-tree-sitter/classes/tree_sitter.Node.html).

For tasks 1, 2, 4, and 5, return a list of violation line numbers in source
order. Return `[]` if there are no violations. Search the whole given subtree.

## Task 1: Braces for If-Statements

Implement `check_if_braces(node)`. Every `if_statement` must have a
node of kind `compound_statement` as its _consequence_ (the body after the condition).
Make the method return the list of lines that contain an `if` with a that has no braces; including nested if-statements.
You only need to check the `if` body; an `else` body must not be checked separately.

Expected result for the example below: `[3, 6]`.

In [ ]:
braces_source = """int adjust(int x) {
    int result = 0;
    if (x > 0)
        result = x;
    if (x < 0) {
        if (x < -10)
            result = -x;
    }
    return result;
}
"""


braces_tree = parser.parse(braces_source.encode("utf-8"))


def check_if_braces(node):
    pass


result = check_if_braces(braces_tree.root_node)
assert result == [3, 6]

## Task 2: Copy-and-Paste Errors

Implement `same_subtree(left, right)` and `check_identical_branches(node)`.
Make the method return the list of lines that contain an `if` where its body and its `else` body are structurally equal.

For subtree equality, compare node kinds, child order, and source text at
leaves. Ignore comment nodes and whitespace between tokens. Consider operators,
identifiers, and literal values. `left == right` checks whether
both refer to the same node, not whether two subtrees have the same structure.

The `alternative` field of an `if_statement` contains an `else_clause`.
Compare its body with the `consequence`, not the `else_clause` wrapper.

Expected result: `[2]`. The first pair of bodies differs only in formatting
and a comment; the second pair assigns different values.

In [ ]:
branches_source = """int choose(int x) {
    if (x > 0) {
        x = 10;
    } else {
        // Same assignment, different formatting.
        x=10;
    }
    if (x < 0) {
        x = 20;
    } else {
        x = 21;
    }
    if (x == 0) {
        x = 1;
    }
    return x;
}
"""


branches_tree = parser.parse(branches_source.encode("utf-8"))


def same_subtree(left, right):
    pass


def check_identical_branches(node):
    pass


result = check_identical_branches(branches_tree.root_node)
assert result == [2]

## Task 3: Function Signatures

Complete `print_declarations_only(node)`. Print the signature of each
**defined function** in source order, including its return type and parameters,
but excluding its body.

Look for `function_definition` nodes and their `body` field.

Expected output:

```text
int add(int left, int right)
static const char *label(int value)
int main(void)
```

In [ ]:
signatures_source = """void log_value(int value);

int add(int left, int right) {
    return left + right;
}

static const char *label(int value) {
    if (value > 0) {
        return "positive";
    }
    return "other";
}

int main(void) {
    log_value(add(2, 3));
    return 0;
}
"""


signatures_tree = parser.parse(signatures_source.encode("utf-8"))


def print_declarations_only(node):
    pass


print_declarations_only(signatures_tree.root_node)

## Task 4: Suspicious Semicolons

Implement `check_empty_if_bodies(node)` for the pattern `if (...); { ... }`.
Make the method return the list of lines that contain an `if` whose _consequence_ is an empty statement (only `;`),
for example `if (x > 0);`

Expected result: `[2]`.

In [ ]:
semicolons_source = """int update(int x) {
    if (x > 0); {
        x = 10;
    }
    if (x < 0) {
        x = -x;
    }
    return x;
}
"""


semicolons_tree = parser.parse(semicolons_source.encode("utf-8"))


def check_empty_if_bodies(node):
    pass


result = check_empty_if_bodies(semicolons_tree.root_node)
assert result == [2]


## Task 5: Redundant Boolean Expressions

Implement `check_redundant_boolean(node)`. Find `binary_expression` nodes
whose `operator` is `||` or `&&` and whose `left` and `right` subtrees are equal.
Reuse `same_subtree` from task 2. Report the starting line of each matching
binary expression.

Expected result: `[3, 6]`. Different literal values or comparison operators
must not count as equal. This is a structural check, not a proof of logical
equivalence; keep parentheses as part of the structure. Treat matches as
suspicious, since repeated expressions with side effects need not be redundant.

In [ ]:
boolean_source = """int classify(int x) {
    int result = 0;
    if (x > 10 || x>10) {
        result += 1;
    }
    if (x < 0 && x < 0) {
        result += 2;
    }
    if (x > 10 || x > 20) {
        result += 4;
    }
    if (x > 10 || x < 10) {
        result += 8;
    }
    return result;
}
"""

boolean_tree = parser.parse(boolean_source.encode("utf-8"))
print(boolean_source)

In [ ]:
def check_redundant_boolean(node):
    pass


print(check_redundant_boolean(boolean_tree.root_node))

---

## Break

Pause here. We continue with nesting and complexity later (if we have time).

---

## Task 6: Cognitive Complexity

Implement `cognitive_complexity(function_node, nesting=0)` and return an integer.
Use this exercise's simplified metric: each `if_statement`, `for_statement`,
`while_statement`, or `do_statement` adds **1 + nesting level**.

The nesting level is the number of enclosing statements of those four kinds
within the function. A top-level `if` or loop has level 0. Blocks and `else`
clauses add no points and do not increase nesting; a statement inside an
`else` body is still inside the enclosing `if`. An `else if` therefore counts
as another nested `if` here. Reset the level for each function.

Use the program below for tasks 6–8.
Expected output:

```text
main: 7
foo: 1
bar: 0
baz: 6
```

In [ ]:
metrics_source = """int main(void) {
    int total = 0;
    for (int i = 0; i < 3; i++) {
        total += i;
    }
    for (int j = 0; j < 3; j++) {
        if (j > 0) {
            total += j;
        } else {
            total -= j;
        }
        if (total > 10) {
            total = 10;
        }
    }
    while (total > 0) {
        total--;
    }
    return total;
}

int foo(int x) {
    if (x > 0) {
        return x;
    } else {
        return -x;
    }
}

void bar(void) {
}

int baz(int x) {
    while (x > 0) {
        if (x > 2) {
            for (int i = 0; i < 2; i++) {
                x--;
            }
        }
        x--;
    }
    return x;
}
"""


metrics_tree = parser.parse(metrics_source.encode("utf-8"))


def cognitive_complexity(function_node, nesting=0):
    pass


for function_node in metrics_tree.root_node.children:
    if function_node.grammar_name == "function_definition":
        declarator = function_node.child_by_field_name("declarator")
        name = declarator.child_by_field_name("declarator").text.decode("utf-8")
        print(f"{name}: {cognitive_complexity(function_node)}")

## Task 7: Deepest Nesting

Implement `deepest_nesting(function_node, nesting=0)`. Return the maximum
number of nested `if` statements and loops along any path in the function.
Use the same statement kinds as in task 6. A function with no such statements
has depth 0; a top-level `if` or loop reaches depth 1. Braces and `else` clauses
do not add a level. Sibling statements do not increase each other's depth.

Expected output for `metrics_source`:

```text
main: 2
foo: 1
bar: 0
baz: 3
```

In [ ]:
def deepest_nesting(function_node, nesting=0):
    pass


for function_node in metrics_tree.root_node.children:
    if function_node.grammar_name == "function_definition":
        declarator = function_node.child_by_field_name("declarator")
        name = declarator.child_by_field_name("declarator").text.decode("utf-8")
        print(f"{name}: {deepest_nesting(function_node)}")

## Task 8: File Outline (For enthusiasts)

Implement `print_outline(node, depth=0)` for `metrics_source`.
Print each defined function's name, followed by its control-flow statements
in source order: `if`, `else`, `for`, `while`, and `do`.
Omit conditions, braces, declarations, expressions, and `return` statements.

Start control-flow statements at depth 1 beneath their function name.
Use `"| "` once per level. Print an `else` at the same depth as its matching
`if`; statements inside either branch go one level deeper.
Functions with no control-flow statements still appear by name.

Expected output:

```text
main
| for
| for
| | if
| | else
| | if
| while
foo
| if
| else
bar
baz
| while
| | if
| | | for
```

In [ ]:
def print_outline(node, depth=0):
    pass


print_outline(metrics_tree.root_node)